# synthetic pipeline condensed-09 11


## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_pipeline_condensed-09_11_code_reference.md`


In [ ]:
import os

import pandas as pd 

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.row_rebuilder import (
    rebuild_consumed_messages_to_observations,
)

from utils.synthetic.pipeline.final_aligned_output_writer import build_synthetic_final_aligned_output_stage

from utils.synthetic.pipeline.final_aligned_incremental import run_final_align_loop



In [ ]:
SCHEMA = os.getenv("CAPSTONE_SCHEMA", "synthetic_run_001")

DATASET_ID = os.getenv("SYNTHETIC_DATASET_ID", "pump_synthetic_v1")
RUN_ID = os.getenv("SYNTHETIC_RUN_ID", "synthetic_run_001")
ASSET_ID = os.getenv("SYNTHETIC_ASSET_ID", "pump_asset_001")

NUMBER_OF_SENSORS = int(52)
IF_EXISTS_FLAG = str("replace")
COMPLETE_ONLY_FLAG = True

BATCH_SIZE = 1000
MAX_ITERATIONS = None   # None = drain everything currently pending
STOP_ON_FAILURE = True

OBSERVATION_WINDOW_SIZE = int(2500)

REBUILD_STATUS = str("pending")
MARK_SOURCE_REBUILT_FLAG = True

REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_consumed_stage")
REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")

PREMELT_TABLE_NAME = "synthetic_observations_premelt_stage"
REBUILT_TABLE_NAME = "synthetic_sensor_observations_rebuilt_stage"
TARGET_TABLE_NAME = "synthetic_sensor_observations_final_aligned_stage"


TIMESTAMP_SOURCE_PRIORITY = (
    "observation_timestamp",
    "timestamp",
    "created_at",
)

STATUS_SOURCE_PRIORITY = (
    "machine_status",
    "stream_state",
    "phase",
)

STATUS_MAPPING = {
    "normal": "NORMAL",
    "broken": "BROKEN",
    "abnormal": "BROKEN",
    "failure": "BROKEN",
    "failed": "BROKEN",
    "fault": "BROKEN",
    "recovering": "RECOVERING",
    "recovery": "RECOVERING",
}


----

In [ ]:
engine = get_engine_from_env()

---

In [ ]:


rebuild_result = rebuild_consumed_messages_to_observations(
    engine=engine,
    schema=SCHEMA,
    source_table=REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME,
    target_table=REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    rebuild_status=REBUILD_STATUS,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    mark_source_rebuilt=MARK_SOURCE_REBUILT_FLAG,
    
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(rebuild_result)

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS rebuilt_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_row_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    """
)

display(validation_dataframe)

---

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id,
        meta_primary_fault_type,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        rebuild_sensor_count,
        rebuild_is_complete
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    ORDER BY observation_index
    LIMIT 10
    """
)
display(sample_dataframe)

---

In [ ]:
incomplete_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        rebuild_sensor_count,
        rebuild_is_complete,
        rebuild_notes
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    WHERE rebuild_is_complete = FALSE
    ORDER BY observation_index
    LIMIT 25
    """
)

display(incomplete_dataframe)

---

In [ ]:
# Dispose Engine
engine.dispose()

---

In [ ]:
engine = get_engine_from_env()

---

In [ ]:
'''

# Full Run

final_output_result = build_synthetic_final_aligned_output_stage(
    engine=engine,
    schema=SCHEMA,
    rebuilt_table=REBUILT_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    if_exists=IF_EXISTS_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
    timestamp_source_priority=TIMESTAMP_SOURCE_PRIORITY,
    status_source_priority=STATUS_SOURCE_PRIORITY,
    status_mapping=STATUS_MAPPING,
    timestamp_output_column="timestamp",
    machine_status_output_column="machine_status",
)

display(final_output_result)

'''

In [ ]:
# Batched Incremental Run

final_align_results = run_final_align_loop(
    engine=engine,
    batch_size=BATCH_SIZE,
    schema=SCHEMA,
    premelt_table=PREMELT_TABLE_NAME,
    rebuilt_table=REBUILT_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    prefer_rebuilt_sensor_values=True,
    max_iterations=MAX_ITERATIONS,
    stop_on_failure=STOP_ON_FAILURE,
)

display(pd.DataFrame(final_align_results))

---

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT dataset_id) AS dataset_id_count,
        COUNT(DISTINCT run_id) AS run_id_count,
        COUNT(DISTINCT asset_id) AS asset_id_count,
        MIN(timestamp) AS min_timestamp,
        MAX(timestamp) AS max_timestamp,
        COUNT(*) FILTER (WHERE machine_status = 'NORMAL') AS normal_rows,
        COUNT(*) FILTER (WHERE machine_status = 'BROKEN') AS broken_rows,
        COUNT(*) FILTER (WHERE machine_status = 'RECOVERING') AS recovering_rows
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    """
)

display(validation_dataframe)


----

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        timestamp,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        machine_status
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    ORDER BY timestamp, dataset_id, run_id, asset_id
    LIMIT 10
    """
)

display(sample_dataframe)


---

In [ ]:
status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        machine_status,
        COUNT(*) AS row_count
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    GROUP BY machine_status
    ORDER BY machine_status
    """
)

display(status_distribution_dataframe)


---

In [ ]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

This condensed support notebook documents a shortened rebuild/final-alignment workflow path.
